# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
from pathlib import Path
from collections import Counter
import re
import sys
import tarfile
import zipfile

import nltk
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 40)

In [2]:
ruta_zip = Path("../txt-files.tar.zip").resolve()
max_documentos = 1000

if not ruta_zip.is_file():
    raise FileNotFoundError(ruta_zip)

try:
    nltk.data.find("tokenizers/punkt_tab/english/")
except LookupError:
    nltk.download("punkt_tab", quiet=True, raise_on_error=True)

In [3]:
def preprocesar(texto):
    tokens = nltk.word_tokenize(texto)
    return [t.lower() for t in tokens if t.isalpha()]


def obtener_titulo(texto, doc_id):
    coincidencia = re.search(
        r"^Title:\s*(.+)", texto, flags=re.MULTILINE | re.IGNORECASE
    )
    return coincidencia.group(1).strip() if coincidencia else f"Gutenberg {doc_id}"

In [4]:
corpus_original = {}
titulos = {}
patron_id = re.compile(r"(\d+)")

with zipfile.ZipFile(ruta_zip) as archivo_zip:
    nombre_tar = next(n for n in archivo_zip.namelist() if n.endswith(".tar"))
    with archivo_zip.open(nombre_tar) as contenido_tar:
        with tarfile.open(fileobj=contenido_tar, mode="r|") as archivo_tar:
            for miembro in archivo_tar:
                if not miembro.isfile() or not miembro.name.endswith(".txt"):
                    continue
                coincidencia = patron_id.search(miembro.name)
                if coincidencia is None:
                    continue
                doc_id = coincidencia.group(1)
                if doc_id in corpus_original:
                    continue
                with archivo_tar.extractfile(miembro) as archivo:
                    texto = archivo.read().decode("utf-8", errors="ignore")
                corpus_original[doc_id] = texto
                titulos[doc_id] = obtener_titulo(texto, doc_id)
                if len(corpus_original) == max_documentos:
                    break

assert len(corpus_original) == max_documentos
print("Documentos cargados:", len(corpus_original))

Documentos cargados: 1000


In [5]:
documentos = pd.DataFrame({
    "Identificador de documento": list(corpus_original),
    "Título": [titulos[d] for d in corpus_original],
    "Texto": list(corpus_original.values())
})
documentos["tokens"] = pd.Series(index=documentos.index, dtype=object)
frecuencias = []

for i, texto in enumerate(documentos["Texto"]):
    tokens = [sys.intern(t) for t in preprocesar(texto)]
    documentos.at[i, "tokens"] = tokens
    frecuencias.append(Counter(tokens))
    if (i + 1) % 100 == 0:
        print("Documentos preprocesados:", i + 1)

del tokens, corpus_original, titulos
display(documentos)

Documentos preprocesados: 100


Documentos preprocesados: 200


Documentos preprocesados: 300


Documentos preprocesados: 400


Documentos preprocesados: 500


Documentos preprocesados: 600


Documentos preprocesados: 700


Documentos preprocesados: 800


Documentos preprocesados: 900


Documentos preprocesados: 1000


,Identificador de documento,Título,Texto,tokens
0,10000,The Magna Carta,The Project Gutenberg eBook of The Magna Carta\r\n \r\nThis eBook is for the use of anyone an...,"[the, project, gutenberg, ebook, of, the, magna, carta, this, ebook, is, for, the, use, of, anyo..."
1,10001,Apocolocyntosis,The Project Gutenberg eBook of Apocolocyntosis\r\n \r\nThis eBook is for the use of anyone an...,"[the, project, gutenberg, ebook, of, apocolocyntosis, this, ebook, is, for, the, use, of, anyone..."
2,10002,The House on the Borderland,The Project Gutenberg eBook of The House on the Borderland\r\n \r\nThis eBook is for the use ...,"[the, project, gutenberg, ebook, of, the, house, on, the, borderland, this, ebook, is, for, the,..."
3,10003,"My First Years as a Frenchwoman, 1876-1879","The Project Gutenberg eBook of My First Years as a Frenchwoman, 1876-1879\r\n \r\nThis eBook ...","[the, project, gutenberg, ebook, of, my, first, years, as, a, frenchwoman, this, ebook, is, for,..."
4,10004,The Warriors,The Project Gutenberg eBook of The Warriors\r\n \r\nThis eBook is for the use of anyone anywh...,"[the, project, gutenberg, ebook, of, the, warriors, this, ebook, is, for, the, use, of, anyone, ..."
...,...,...,...,...
995,11079,Essays in Rebellion,The Project Gutenberg eBook of Essays in Rebellion\r\n \r\nThis eBook is for the use of anyon...,"[the, project, gutenberg, ebook, of, essays, in, rebellion, this, ebook, is, for, the, use, of, ..."
996,1107,The Taming of the Shrew,The Project Gutenberg eBook of The Taming of the Shrew\r\n \r\nThis eBook is for the use of a...,"[the, project, gutenberg, ebook, of, the, taming, of, the, shrew, this, ebook, is, for, the, use..."
997,11080,"The Orations of Marcus Tullius Cicero, Volume 4","The Project Gutenberg eBook of The Orations of Marcus Tullius Cicero, Volume 4\r\n \r\nThis e...","[the, project, gutenberg, ebook, of, the, orations, of, marcus, tullius, cicero, volume, this, e..."
998,11081,Extracto de la gramatica mutsun,The Project Gutenberg eBook of Extracto de la gramatica mutsun\r\n \r\nThis eBook is for the ...,"[the, project, gutenberg, ebook, of, extracto, de, la, gramatica, mutsun, this, ebook, is, for, ..."


In [6]:
ids_documentos = documentos["Identificador de documento"].astype(str).to_numpy()
N = len(documentos)
longitudes = documentos["tokens"].map(len).to_numpy()
longitud_media = longitudes.mean()
df_corpus = Counter()

for frecuencia in frecuencias:
    df_corpus.update(frecuencia.keys())

vocabulario = sorted(df_corpus)
indice_terminos = {t: j for j, t in enumerate(vocabulario)}
df_vector = np.array([df_corpus[t] for t in vocabulario])
idf_tfidf = np.log(N / df_vector) + 1
idf_bm25 = np.log(1 + (N - df_vector + 0.5) / (df_vector + 0.5))

estadisticas_terminos = pd.DataFrame({
    "Término": vocabulario,
    "DF": df_vector,
    "IDF TF-IDF": idf_tfidf,
    "IDF BM25": idf_bm25
})
display(estadisticas_terminos)
display(pd.DataFrame([{
    "Documentos": N,
    "Vocabulario": len(vocabulario),
    "Longitud mínima": longitudes.min(),
    "Longitud media": longitud_media,
    "Longitud máxima": longitudes.max()
}]))

,Término,DF,IDF TF-IDF,IDF BM25
0,a,1000,1.000000,0.000500
1,aa,18,5.017384,3.990984
2,aaa,4,6.521461,5.404677
3,aaaaaah,2,7.214608,5.992464
4,aaaarranger,1,7.907755,6.503290
...,...,...,...,...
611587,ℨß,1,7.907755,6.503290
611588,丞,1,7.907755,6.503290
611589,卍,1,7.907755,6.503290
611590,相,1,7.907755,6.503290


,Documentos,Vocabulario,Longitud mínima,Longitud media,Longitud máxima
0,1000,611592,3118,70157.368,976132


Se conserva la primera versión TXT de los mismos 1.000 identificadores de la práctica 3. La tokenización usa `nltk.word_tokenize`, conserva palabras alfabéticas y las convierte a minúsculas, sin eliminar stopwords ni aplicar stemming. La colección contiene varios idiomas; las consultas se formulan en inglés, presente en los documentos utilizados.

Se utilizan $IDF_{TFIDF}(t)=\ln(N/df(t))+1$ e $IDF_{BM25}(t)=\ln(1+(N-df(t)+0.5)/(df(t)+0.5))$. Las estadísticas se calculan una sola vez sobre toda la colección.

**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [7]:
consultas = pd.DataFrame([
    ("Q1", "the", "Explorar la recuperación con una palabra funcional muy frecuente."),
    ("Q2", "love marriage family", "Encontrar textos sobre relaciones de pareja y vida familiar."),
    ("Q3", "sea ship captain whale", "Encontrar relatos de navegación, capitanes y ballenas."),
    ("Q4", "science evolution species nature", "Encontrar textos sobre evolución y diversidad de especies."),
    ("Q5", "war army battle soldier", "Encontrar relatos de campañas militares y vida de soldados."),
    ("Q6", "apocolocyntosis", "Localizar la obra Apocolocyntosis."),
    ("Q7", "spermaceti whale", "Encontrar información sobre ballenas y espermaceti."),
    ("Q8", "telescope astronomy planets stars", "Encontrar textos sobre astronomía y observación del cielo.")
], columns=["Consulta", "Texto", "Necesidad de información"])
consultas["tokens"] = consultas["Texto"].map(preprocesar)
consultas["Longitud"] = consultas["tokens"].map(len)
display(consultas)

,Consulta,Texto,Necesidad de información,tokens,Longitud
0,Q1,the,Explorar la recuperación con una palabra funcional muy frecuente.,[the],1
1,Q2,love marriage family,Encontrar textos sobre relaciones de pareja y vida familiar.,"[love, marriage, family]",3
2,Q3,sea ship captain whale,"Encontrar relatos de navegación, capitanes y ballenas.","[sea, ship, captain, whale]",4
3,Q4,science evolution species nature,Encontrar textos sobre evolución y diversidad de especies.,"[science, evolution, species, nature]",4
4,Q5,war army battle soldier,Encontrar relatos de campañas militares y vida de soldados.,"[war, army, battle, soldier]",4
5,Q6,apocolocyntosis,Localizar la obra Apocolocyntosis.,[apocolocyntosis],1
6,Q7,spermaceti whale,Encontrar información sobre ballenas y espermaceti.,"[spermaceti, whale]",2
7,Q8,telescope astronomy planets stars,Encontrar textos sobre astronomía y observación del cielo.,"[telescope, astronomy, planets, stars]",4


In [8]:
frecuencias_consultas = []
for _, consulta in consultas.iterrows():
    for termino in dict.fromkeys(consulta["tokens"]):
        df = df_corpus.get(termino, 0)
        frecuencia = "Frecuente" if df >= 0.5 * N else "Poco frecuente" if df <= 0.05 * N else "Intermedia"
        frecuencias_consultas.append({
            "Consulta": consulta["Consulta"],
            "Término": termino,
            "DF": df,
            "DF / N": df / N,
            "Frecuencia documental": frecuencia
        })

display(pd.DataFrame(frecuencias_consultas))
print("DF de 'the':", df_corpus.get("the", 0))
print("DF de 'apocolocyntosis':", df_corpus.get("apocolocyntosis", 0))
print("Longitudes de las consultas:", sorted(consultas["Longitud"].unique()))

,Consulta,Término,DF,DF / N,Frecuencia documental
0,Q1,the,1000,1.000,Frecuente
1,Q2,love,823,0.823,Frecuente
2,Q2,marriage,559,0.559,Frecuente
3,Q2,family,784,0.784,Frecuente
4,Q3,sea,789,0.789,Frecuente
5,Q3,ship,600,0.600,Frecuente
6,Q3,captain,539,0.539,Frecuente
7,Q3,whale,132,0.132,Intermedia
8,Q4,science,518,0.518,Frecuente
9,Q4,evolution,147,0.147,Intermedia


DF de 'the': 1000
DF de 'apocolocyntosis': 1
Longitudes de las consultas: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [9]:
def conservar_tokens(tokens):
    return tokens


vectorizador_tf = CountVectorizer(
    analyzer=conservar_tokens,
    vocabulary=indice_terminos,
    lowercase=False,
    dtype=np.float64
)
matriz_tf = vectorizador_tf.fit_transform(documentos["tokens"]).tocsr()
np.testing.assert_array_equal(
    np.asarray((matriz_tf > 0).sum(axis=0)).ravel(), df_vector
)
matriz_tfidf = normalize(matriz_tf.multiply(idf_tfidf).tocsr(), norm="l2")
matriz_tf_columnas = matriz_tf.tocsc()

print("Dimensiones:", matriz_tfidf.shape)
print("Formato de TF-IDF:", matriz_tfidf.format)
print("Valores distintos de cero:", matriz_tfidf.nnz)
print("Memoria de TF-IDF (MiB):", round((
    matriz_tfidf.data.nbytes + matriz_tfidf.indices.nbytes + matriz_tfidf.indptr.nbytes
) / 2**20, 2))

Dimensiones: (1000, 611592)
Formato de TF-IDF: csr
Valores distintos de cero: 6597387
Memoria de TF-IDF (MiB): 75.5


In [10]:
def puntuar_tfidf(tokens_consulta):
    tf_consulta = vectorizador_tf.transform([tokens_consulta])
    vector_consulta = normalize(tf_consulta.multiply(idf_tfidf).tocsr(), norm="l2")
    return (matriz_tfidf @ vector_consulta.T).toarray().ravel()


def puntuar_bm25(tokens_consulta, k1=1.2, b=0.75):
    puntuaciones = np.zeros(N)
    ajuste = k1 * (1 - b + b * longitudes / longitud_media)
    for termino in sorted(set(tokens_consulta)):
        if termino not in indice_terminos:
            continue
        j = indice_terminos[termino]
        columna = matriz_tf_columnas.getcol(j)
        filas = columna.indices
        tf = columna.data
        puntuaciones[filas] += idf_bm25[j] * tf * (k1 + 1) / (tf + ajuste[filas])
    return puntuaciones

In [11]:
def ordenar_documentos(puntuaciones):
    positivos = np.flatnonzero(puntuaciones > 0)
    orden = np.lexsort((ids_documentos[positivos], -puntuaciones[positivos]))
    return positivos[orden]


def construir_ranking(puntuaciones, limite=10):
    orden = ordenar_documentos(puntuaciones)[:limite]
    return pd.DataFrame({
        "Posición": np.arange(1, len(orden) + 1),
        "Identificador": ids_documentos[orden],
        "Título": documentos.iloc[orden]["Título"].to_numpy(),
        "Puntuación": puntuaciones[orden]
    })

In [12]:
k1 = 1.2
b = 0.75
resultados = {}

for _, consulta in consultas.iterrows():
    scores_tfidf = puntuar_tfidf(consulta["tokens"])
    scores_bm25 = puntuar_bm25(consulta["tokens"], k1=k1, b=b)
    resultados[consulta["Consulta"]] = {
        "TF-IDF": scores_tfidf,
        "BM25": scores_bm25,
        "Top TF-IDF": construir_ranking(scores_tfidf),
        "Top BM25": construir_ranking(scores_bm25)
    }
    print(consulta["Consulta"], "—", consulta["Texto"])
    print("TF-IDF + coseno")
    display(resultados[consulta["Consulta"]]["Top TF-IDF"])
    print("BM25")
    display(resultados[consulta["Consulta"]]["Top BM25"])

Q1 — the
TF-IDF + coseno


,Posición,Identificador,Título,Puntuación
0,1,10702,"The History of Rome, Book II",0.786110
1,2,10908,"Manual for Noncommissioned Officers and Privates of Infantry of the Army of the United States, 1917",0.785983
2,3,10701,"The History of Rome, Book I",0.779282
3,4,10706,The History of Rome (Volumes 1-5),0.779268
4,5,10704,"The History of Rome, Book IV",0.778149
5,6,10726,"Outlines of Lessons in Botany, Part I; from Seed to Leaf",0.767262
6,7,10703,"The History of Rome, Book III",0.765810
7,8,10705,"The History of Rome, Book V",0.763894
8,9,10453,A Practical Physiology: A Text-Book for Higher Schools,0.762729
9,10,10795,Yorkshire—Coast and Moorland Scenes,0.762255


BM25


,Posición,Identificador,Título,Puntuación
0,1,10706,The History of Rome (Volumes 1-5),0.001099
1,2,10998,A Catechism of the Steam Engine,0.001099
2,3,10704,"The History of Rome, Book IV",0.001099
3,4,10705,"The History of Rome, Book V",0.001099
4,5,10703,"The History of Rome, Book III",0.001099
5,6,11034,"A Compilation of the Messages and Papers of the Presidents. Volume 3, part 2: Martin Van Buren",0.001099
6,7,10702,"The History of Rome, Book II",0.001099
7,8,10701,"The History of Rome, Book I",0.001099
8,9,10919,"A Compilation of the Messages and Papers of the Presidents. Volume 2, part 1: James Monroe",0.001099
9,10,10453,A Practical Physiology: A Text-Book for Higher Schools,0.001099


Q2 — love marriage family
TF-IDF + coseno


,Posición,Identificador,Título,Puntuación
0,1,1105,The Sonnets,0.064203
1,2,1041,Shakespeare's Sonnets,0.062387
2,3,10063,Happiness and Marriage,0.046531
3,4,10913,English Poems,0.045026
4,5,1045,Venus and Adonis,0.040304
5,6,10740,The Way of Peace,0.038117
6,7,10910,"Robert Louis Stevenson, an Elegy; and Other Poems",0.034120
7,8,10495,Under King Constantine,0.031140
8,9,10665,"Life, letters, and Epicurean philosophy of Ninon de L'Enclos, the celebrated beauty of the seven...",0.030966
9,10,10133,"The Vigil of Venus and Other Poems by ""Q""",0.029852


... (22 outputs left out)

Q8 — telescope astronomy planets stars
TF-IDF + coseno


,Posición,Identificador,Título,Puntuación
0,1,10202,"Maria Mitchell: Life, Letters, and Journals",0.020417
1,2,10655,Autobiography of Sir George Biddell Airy,0.018231
2,3,10855,"Is Mars habitable? A critical examination of Professor Percival Lowell's book ""Mars and its cana...",0.012790
3,4,10773,Ancient and Modern Physics,0.012591
4,5,10484,"Beacon Lights of History, Volume 03: Ancient Achievements",0.007372
5,6,10811,De La Salle Fifth Reader,0.006212
6,7,10005,A Voyage to the Moon,0.006116
7,8,10913,English Poems,0.005872
8,9,10532,"Beacon Lights of History, Volume 06: Renaissance and Reformation",0.005115
9,10,10088,"Thaumaturgia; Or, Elucidations of the Marvellous",0.004917


BM25


,Posición,Identificador,Título,Puntuación
0,1,10655,Autobiography of Sir George Biddell Airy,14.751343
1,2,10202,"Maria Mitchell: Life, Letters, and Journals",14.455500
2,3,10532,"Beacon Lights of History, Volume 06: Renaissance and Reformation",13.331440
3,4,10855,"Is Mars habitable? A critical examination of Professor Percival Lowell's book ""Mars and its cana...",12.893907
4,5,10898,"American Scenes, and Christian Slavery",11.193061
5,6,1013,The First Men in the Moon,10.308649
6,7,10093,Gutta-Percha Willie,9.916322
7,8,10128,"The Great Events by Famous Historians, Volume 17",9.728000
8,9,10484,"Beacon Lights of History, Volume 03: Ancient Achievements",9.575904
9,10,10427,Scientific Essays and Lectures,9.535706


In [13]:
termino_ausente = "zzzinexistentegutenbergzzz"
while termino_ausente in indice_terminos:
    termino_ausente += "z"

tokens_ausentes = preprocesar(termino_ausente)
for funcion in [puntuar_tfidf, puntuar_bm25]:
    puntuaciones = funcion(tokens_ausentes)
    assert np.all(puntuaciones == 0)
    assert construir_ranking(puntuaciones).empty

orden_empate = ordenar_documentos(np.ones(N))
assert ids_documentos[orden_empate].tolist() == sorted(ids_documentos.tolist())
print("Consulta sin términos conocidos: ambos rankings vacíos.")
print("Desempates verificados por identificador en orden lexicográfico.")

Consulta sin términos conocidos: ambos rankings vacíos.
Desempates verificados por identificador en orden lexicográfico.


Las consultas tienen uno, dos, tres y cuatro términos. La tabla de DF permite contrastar palabras frecuentes como `the` con términos específicos como `apocolocyntosis` y `spermaceti`. Los umbrales de la tabla son 50 % para frecuencia alta y 5 % para frecuencia baja; no se utilizan para filtrar palabras.

TF-IDF usa los IDF del corpus y normalización L2 de documentos y consultas. BM25 suma una vez por término distinto de consulta:

$$s(q,d)=\sum_{t\in q}IDF_{BM25}(t)\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/avgdl)}.$$

Los dos métodos comparten exactamente los documentos y tokens. Solo se presentan puntuaciones positivas; las estadísticas permanecen fijas al cambiar de consulta.

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [14]:
resumen_rankings = []

for _, consulta in consultas.iterrows():
    clave = consulta["Consulta"]
    tfidf = resultados[clave]["Top TF-IDF"]
    bm25 = resultados[clave]["Top BM25"]
    tabla_tfidf = tfidf[["Posición", "Puntuación"]].rename(columns={"Puntuación": "Coseno"}).copy()
    tabla_tfidf["ID y título TF-IDF"] = tfidf["Identificador"] + " — " + tfidf["Título"]
    tabla_bm25 = bm25[["Posición", "Puntuación"]].rename(columns={"Puntuación": "Score BM25"}).copy()
    tabla_bm25["ID y título BM25"] = bm25["Identificador"] + " — " + bm25["Título"]
    comparacion = tabla_tfidf.merge(tabla_bm25, on="Posición", how="outer").sort_values("Posición")
    print(clave, "—", consulta["Texto"])
    display(comparacion[["Posición", "ID y título TF-IDF", "Coseno", "ID y título BM25", "Score BM25"]])
    resumen_rankings.append({
        "Consulta": clave,
        "Tamaño TF-IDF": len(tfidf),
        "Tamaño BM25": len(bm25),
        "IDs compartidos": len(set(tfidf["Identificador"]) & set(bm25["Identificador"])),
        "Primero TF-IDF": tfidf.iloc[0]["Identificador"] if len(tfidf) else None,
        "Primero BM25": bm25.iloc[0]["Identificador"] if len(bm25) else None
    })

display(pd.DataFrame(resumen_rankings))

Q1 — the


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,"10702 — The History of Rome, Book II",0.786110,10706 — The History of Rome (Volumes 1-5),0.001099
1,2,10908 — Manual for Noncommissioned Officers and Privates of Infantry of the Army of the United S...,0.785983,10998 — A Catechism of the Steam Engine,0.001099
2,3,"10701 — The History of Rome, Book I",0.779282,"10704 — The History of Rome, Book IV",0.001099
3,4,10706 — The History of Rome (Volumes 1-5),0.779268,"10705 — The History of Rome, Book V",0.001099
4,5,"10704 — The History of Rome, Book IV",0.778149,"10703 — The History of Rome, Book III",0.001099
5,6,"10726 — Outlines of Lessons in Botany, Part I; from Seed to Leaf",0.767262,"11034 — A Compilation of the Messages and Papers of the Presidents. Volume 3, part 2: Martin Van...",0.001099
6,7,"10703 — The History of Rome, Book III",0.765810,"10702 — The History of Rome, Book II",0.001099
7,8,"10705 — The History of Rome, Book V",0.763894,"10701 — The History of Rome, Book I",0.001099
8,9,10453 — A Practical Physiology: A Text-Book for Higher Schools,0.762729,"10919 — A Compilation of the Messages and Papers of the Presidents. Volume 2, part 1: James Monroe",0.001099
9,10,10795 — Yorkshire—Coast and Moorland Scenes,0.762255,10453 — A Practical Physiology: A Text-Book for Higher Schools,0.001099


Q2 — love marriage family


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,1105 — The Sonnets,0.064203,10063 — Happiness and Marriage,2.205607
1,2,1041 — Shakespeare's Sonnets,0.062387,"10957 — The Love Affairs of Great Musicians, Volume 1",2.199886
2,3,10063 — Happiness and Marriage,0.046531,10877 — The Tragedies of the Medici,2.193155
3,4,10913 — English Poems,0.045026,10471 — The World's Greatest Books — Volume 01 — Fiction,2.189496
4,5,1045 — Venus and Adonis,0.040304,10921 — The World's Greatest Books — Volume 04 — Fiction,2.185086
5,6,10740 — The Way of Peace,0.038117,"10533 — Beacon Lights of History, Volume 07: Great Women",2.182522
6,7,"10910 — Robert Louis Stevenson, an Elegy; and Other Poems",0.034120,10799 — Clarissa Harlowe; or the history of a young lady — Volume 5,2.182365
7,8,10495 — Under King Constantine,0.031140,"10590 — Lady Mary Wortley Montague, Her Life and Letters (1689-1762)",2.175522
8,9,"10665 — Life, letters, and Epicurean philosophy of Ninon de L'Enclos, the celebrated beauty of t...",0.030966,10365 — Precaution: A Novel,2.166969
9,10,"10133 — The Vigil of Venus and Other Poems by ""Q""",0.029852,"11057 — The Wife of his Youth and Other Stories of the Color Line, and Selected Essays",2.166370


Q3 — sea ship captain whale


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,10617 — Within the Deep,0.056982,"10545 — The Sea Lions; Or, The Lost Sealers",7.114219
1,2,"10871 — At Sunwich Port, Part 1.",0.047632,10765 — Thrilling Adventures by Land and Sea,7.097235
2,3,10339 — An Antarctic Mystery,0.044893,"10997 — The World of Waters, Or, A Peaceful Progress O'er the Unpathed Sea",7.052598
3,4,1058 — The Mirror of the Sea,0.040450,"10224 — Kalitan, Our Little Alaskan Cousin",6.728314
4,5,10008 — The Mystery,0.036873,10599 — Lost in the Air,6.704909
5,6,"10712 — White Jacket; Or, The World on a Man-of-War",0.034254,"10848 — Natalie; Or, A Gem Among the Sea-Weeds",6.538901
6,7,10370 — Sustained honor: The Age of Liberty Established,0.034244,10600 — A General History and Collection of Voyages and Travels — Volume 01,6.426570
7,8,10876 — At Sunwich Port [complete],0.030919,1082 — Voyage of the Paper Canoe,6.398855
8,9,"10874 — At Sunwich Port, Part 4.",0.030669,"10015 — Punchinello, Volume 1, No. 19, August 6, 1870",6.395135
9,10,"10842 — The Life of Captain James Cook, the Circumnavigator",0.028233,10339 — An Antarctic Mystery,6.315032


... (8 outputs left out)

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,10617 — Within the Deep,0.025899,"10545 — The Sea Lions; Or, The Lost Sealers",12.622837
1,2,"10015 — Punchinello, Volume 1, No. 19, August 6, 1870",0.011019,"10952 — Punch, or the London Charivari, Volume 156, Jan. 15, 1919",9.772021
2,3,10040 — Aleutian Indian and English Dictionary,0.009507,"10997 — The World of Waters, Or, A Peaceful Progress O'er the Unpathed Sea",8.169772
3,4,"10545 — The Sea Lions; Or, The Lost Sealers",0.006993,10766 — Enquire within upon everything,8.020001
4,5,10430 — Trips to the Moon,0.006729,10681 — Roget's Thesaurus of English Words and Phrases,6.056252
5,6,"10979 — Footsteps on the Road to Learning; Or, The Alphabet in Rhyme",0.006585,10136 — The Book of Household Management,5.952074
6,7,"10224 — Kalitan, Our Little Alaskan Cousin",0.006437,10699 — Character Writings of the Seventeenth Century,4.981764
7,8,10122 — Fairies and Fusiliers,0.005686,"10383 — The maternal management of children, in health and disease",4.981497
8,9,10599 — Lost in the Air,0.005064,10617 — Within the Deep,4.383887
9,10,"10952 — Punch, or the London Charivari, Volume 156, Jan. 15, 1919",0.004079,"10015 — Punchinello, Volume 1, No. 19, August 6, 1870",4.269393


Q8 — telescope astronomy planets stars


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,"10202 — Maria Mitchell: Life, Letters, and Journals",0.020417,10655 — Autobiography of Sir George Biddell Airy,14.751343
1,2,10655 — Autobiography of Sir George Biddell Airy,0.018231,"10202 — Maria Mitchell: Life, Letters, and Journals",14.455500
2,3,"10855 — Is Mars habitable? A critical examination of Professor Percival Lowell's book ""Mars and ...",0.012790,"10532 — Beacon Lights of History, Volume 06: Renaissance and Reformation",13.331440
3,4,10773 — Ancient and Modern Physics,0.012591,"10855 — Is Mars habitable? A critical examination of Professor Percival Lowell's book ""Mars and ...",12.893907
4,5,"10484 — Beacon Lights of History, Volume 03: Ancient Achievements",0.007372,"10898 — American Scenes, and Christian Slavery",11.193061
5,6,10811 — De La Salle Fifth Reader,0.006212,1013 — The First Men in the Moon,10.308649
6,7,10005 — A Voyage to the Moon,0.006116,10093 — Gutta-Percha Willie,9.916322
7,8,10913 — English Poems,0.005872,"10128 — The Great Events by Famous Historians, Volume 17",9.728000
8,9,"10532 — Beacon Lights of History, Volume 06: Renaissance and Reformation",0.005115,"10484 — Beacon Lights of History, Volume 03: Ancient Achievements",9.575904
9,10,"10088 — Thaumaturgia; Or, Elucidations of the Marvellous",0.004917,10427 — Scientific Essays and Lectures,9.535706


,Consulta,Tamaño TF-IDF,Tamaño BM25,IDs compartidos,Primero TF-IDF,Primero BM25
0,Q1,10,10,7,10702,10706
1,Q2,10,10,1,1105,10063
2,Q3,10,10,1,10617,10545
3,Q4,10,10,5,10852,10139
4,Q5,10,10,5,10629,10692
5,Q6,1,1,1,10001,10001
6,Q7,10,10,4,10617,10545
7,Q8,10,10,5,10202,10655


In [15]:
consulta_elegida = consultas.set_index("Consulta").loc["Q3"]
tokens_elegidos = consulta_elegida["tokens"]
scores_tfidf = resultados["Q3"]["TF-IDF"]
scores_bm25 = resultados["Q3"]["BM25"]
orden_tfidf = ordenar_documentos(scores_tfidf)
orden_bm25 = ordenar_documentos(scores_bm25)
posiciones_tfidf = {int(i): p for p, i in enumerate(orden_tfidf, start=1)}
posiciones_bm25 = {int(i): p for p, i in enumerate(orden_bm25, start=1)}
candidatos = list(dict.fromkeys(orden_bm25[:10].tolist() + orden_tfidf[:10].tolist()))

pares = []
for i in candidatos:
    for j in candidatos:
        if i >= j or i not in posiciones_tfidf or j not in posiciones_tfidf:
            continue
        if i not in posiciones_bm25 or j not in posiciones_bm25:
            continue
        cambio = (posiciones_tfidf[i] - posiciones_tfidf[j]) * (posiciones_bm25[i] - posiciones_bm25[j])
        distinta_tf = any(frecuencias[i].get(t, 0) != frecuencias[j].get(t, 0) for t in tokens_elegidos)
        if cambio < 0 and longitudes[i] != longitudes[j] and distinta_tf:
            pares.append((i, j))

assert pares, "No se encontró un par con inversión de posiciones."
seleccionados = min(pares, key=lambda par: posiciones_bm25[par[0]] + posiciones_bm25[par[1]])
print("Consulta elegida: Q3 —", consulta_elegida["Texto"])

tabla_seleccionados = pd.DataFrame([{
    "Identificador": ids_documentos[i],
    "Título": documentos.iloc[i]["Título"],
    "Longitud": longitudes[i],
    "Longitud / media": longitudes[i] / longitud_media,
    "Posición TF-IDF": posiciones_tfidf[i],
    "Coseno": scores_tfidf[i],
    "Posición BM25": posiciones_bm25[i],
    "Score BM25": scores_bm25[i]
} for i in seleccionados])
display(tabla_seleccionados)

Consulta elegida: Q3 — sea ship captain whale


,Identificador,Título,Longitud,Longitud / media,Posición TF-IDF,Coseno,Posición BM25,Score BM25
0,10545,"The Sea Lions; Or, The Lost Sealers",170483,2.430008,24,0.019250,1,7.114219
1,10997,"The World of Waters, Or, A Peaceful Progress O'er the Unpathed Sea",94252,1.343437,15,0.024284,3,7.052598


In [16]:
def aportes_bm25(tokens_consulta, indices_documentos, k1=1.2, b=0.75):
    filas = []
    for i in indices_documentos:
        ajuste = 1 - b + b * longitudes[i] / longitud_media
        for termino in sorted(set(tokens_consulta)):
            j = indice_terminos.get(termino)
            tf = frecuencias[i].get(termino, 0)
            idf = idf_bm25[j] if j is not None else 0.0
            factor_tf = tf * (k1 + 1) / (tf + k1 * ajuste) if tf else 0.0
            filas.append({
                "Identificador": ids_documentos[i],
                "Término": termino,
                "Longitud": longitudes[i],
                "f(t,d)": tf,
                "df(t)": df_corpus.get(termino, 0),
                "IDF BM25": idf,
                "Ajuste longitud": ajuste,
                "Factor TF saturado": factor_tf,
                "Aporte BM25": idf * factor_tf
            })
    return pd.DataFrame(filas)

In [17]:
detalle = aportes_bm25(tokens_elegidos, seleccionados, k1=k1, b=b)
vector_q = normalize(
    vectorizador_tf.transform([tokens_elegidos]).multiply(idf_tfidf).tocsr(), norm="l2"
)
aportes_coseno = {}
for i in seleccionados:
    for t in sorted(set(tokens_elegidos)):
        j = indice_terminos[t]
        aportes_coseno[(ids_documentos[i], t)] = vector_q[0, j] * matriz_tfidf[i, j]

detalle["Aporte coseno"] = [
    aportes_coseno[(fila["Identificador"], fila["Término"])]
    for _, fila in detalle.iterrows()
]
display(detalle)

for i in seleccionados:
    filas = detalle[detalle["Identificador"] == ids_documentos[i]]
    np.testing.assert_allclose(filas["Aporte BM25"].sum(), scores_bm25[i])
    np.testing.assert_allclose(filas["Aporte coseno"].sum(), scores_tfidf[i])
print("La suma de aportes coincide con la puntuación de cada método.")

,Identificador,Término,Longitud,"f(t,d)",df(t),IDF BM25,Ajuste longitud,Factor TF saturado,Aporte BM25,Aporte coseno
0,10545,captain,170483,205,539,0.618112,2.072506,2.173630,1.343547,0.006908
1,10545,sea,170483,253,789,0.237355,2.072506,2.178584,0.517098,0.004983
2,10545,ship,170483,18,600,0.510992,2.072506,1.932932,0.987713,0.000529
3,10545,whale,170483,58,132,2.022172,2.072506,2.109544,4.265861,0.006831
4,10997,captain,94252,106,539,0.618112,1.257578,2.169119,1.340758,0.006666
5,10997,sea,94252,233,789,0.237355,1.257578,2.185843,0.518821,0.008564
6,10997,ship,94252,97,600,0.510992,1.257578,2.166297,1.106961,0.005318
7,10997,whale,94252,17,132,2.022172,1.257578,2.020628,4.086059,0.003736


La suma de aportes coincide con la puntuación de cada método.


In [18]:
mejor_tfidf = min(seleccionados, key=lambda i: posiciones_tfidf[i])
mejor_bm25 = min(seleccionados, key=lambda i: posiciones_bm25[i])
mas_corto = min(seleccionados, key=lambda i: longitudes[i])
diferencias = detalle.pivot(index="Término", columns="Identificador", values="Aporte BM25")
otro = next(i for i in seleccionados if i != mejor_bm25)
diferencia_aportes = diferencias[ids_documentos[mejor_bm25]] - diferencias[ids_documentos[otro]]
termino_decisivo = diferencia_aportes.idxmax()

display(Markdown(
    f"En Q3, TF-IDF + coseno coloca **{ids_documentos[mejor_tfidf]}** antes que "
    f"**{ids_documentos[mejor_bm25]}**: sus cosenos son "
    f"{scores_tfidf[mejor_tfidf]:.6f} y {scores_tfidf[mejor_bm25]:.6f}. "
    f"BM25 invierte ese orden: {scores_bm25[mejor_bm25]:.6f} para "
    f"**{ids_documentos[mejor_bm25]}** frente a {scores_bm25[mejor_tfidf]:.6f} para "
    f"**{ids_documentos[mejor_tfidf]}**.\n\n"
    f"El documento más corto es **{ids_documentos[mas_corto]}**, con "
    f"{longitudes[mas_corto]:,} tokens, frente a una media de {longitud_media:,.2f}. "
    f"La mayor diferencia de aportes a favor del documento preferido por BM25 corresponde a "
    f"**{termino_decisivo}**, con df = {df_corpus[termino_decisivo]}: aparece "
    f"{frecuencias[mejor_bm25].get(termino_decisivo, 0)} veces en "
    f"{ids_documentos[mejor_bm25]} y {frecuencias[otro].get(termino_decisivo, 0)} en "
    f"{ids_documentos[otro]}.\n\n"
    "La tabla separa frecuencia, rareza y ajuste por longitud. El IDF de un término es el mismo "
    "para ambos documentos; su aporte cambia por TF y longitud. La saturación limita el beneficio "
    "de repetir un término. En coseno, los aportes corresponden al producto de pesos TF-IDF "
    "normalizados por el vector completo. Las puntuaciones de cada método explican su propio "
    "orden; sus escalas no se comparan entre sí. La coincidencia de IDs no mide relevancia "
    "sin una evaluación con juicios de relevancia."
))

En Q3, TF-IDF + coseno coloca **10997** antes que **10545**: sus cosenos son 0.024284 y 0.019250. BM25 invierte ese orden: 7.114219 para **10545** frente a 7.052598 para **10997**.

El documento más corto es **10997**, con 94,252 tokens, frente a una media de 70,157.37. La mayor diferencia de aportes a favor del documento preferido por BM25 corresponde a **whale**, con df = 132: aparece 58 veces en 10545 y 17 en 10997.

La tabla separa frecuencia, rareza y ajuste por longitud. El IDF de un término es el mismo para ambos documentos; su aporte cambia por TF y longitud. La saturación limita el beneficio de repetir un término. En coseno, los aportes corresponden al producto de pesos TF-IDF normalizados por el vector completo. Las puntuaciones de cada método explican su propio orden; sus escalas no se comparan entre sí. La coincidencia de IDs no mide relevancia sin una evaluación con juicios de relevancia.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [19]:
configuraciones = [
    ("Base", 1.2, 0.75),
    ("Sin ajuste por longitud", 1.2, 0.0),
    ("Mayor efecto de las repeticiones", 2.0, 0.75)
]
resultados_parametros = []
aportes_parametros = []

for nombre, k1_actual, b_actual in configuraciones:
    puntuaciones = puntuar_bm25(tokens_elegidos, k1=k1_actual, b=b_actual)
    orden = ordenar_documentos(puntuaciones)
    posiciones = {int(i): p for p, i in enumerate(orden, start=1)}
    for i in seleccionados:
        resultados_parametros.append({
            "Configuración": nombre,
            "k1": k1_actual,
            "b": b_actual,
            "Identificador": ids_documentos[i],
            "Longitud": longitudes[i],
            "Posición": posiciones[i],
            "Cambio de posición": posiciones[i] - posiciones_bm25[i],
            "Score BM25": puntuaciones[i],
            "Cambio de score": puntuaciones[i] - scores_bm25[i]
        })
    aportes = aportes_bm25(tokens_elegidos, seleccionados, k1=k1_actual, b=b_actual)
    aportes.insert(0, "Configuración", nombre)
    for i in seleccionados:
        np.testing.assert_allclose(
            aportes.loc[aportes["Identificador"] == ids_documentos[i], "Aporte BM25"].sum(),
            puntuaciones[i]
        )
    aportes_parametros.append(aportes)

tabla_parametros = pd.DataFrame(resultados_parametros)
tabla_aportes = pd.concat(aportes_parametros, ignore_index=True)
display(tabla_parametros)
display(tabla_aportes)

,Configuración,k1,b,Identificador,Longitud,Posición,Cambio de posición,Score BM25,Cambio de score
0,Base,1.2,0.75,10545,170483,1,0,7.114219,0.000000
1,Base,1.2,0.75,10997,94252,3,0,7.052598,0.000000
2,Sin ajuste por longitud,1.2,0.00,10545,170483,1,0,7.284170,0.169952
3,Sin ajuste por longitud,1.2,0.00,10997,94252,3,0,7.130027,0.077429
4,Mayor efecto de las repeticiones,2.0,0.75,10545,170483,1,0,9.426098,2.311879
5,Mayor efecto de las repeticiones,2.0,0.75,10997,94252,3,0,9.294699,2.242101


,Configuración,Identificador,Término,Longitud,"f(t,d)",df(t),IDF BM25,Ajuste longitud,Factor TF saturado,Aporte BM25
0,Base,10545,captain,170483,205,539,0.618112,2.072506,2.173630,1.343547
1,Base,10545,sea,170483,253,789,0.237355,2.072506,2.178584,0.517098
2,Base,10545,ship,170483,18,600,0.510992,2.072506,1.932932,0.987713
3,Base,10545,whale,170483,58,132,2.022172,2.072506,2.109544,4.265861
4,Base,10997,captain,94252,106,539,0.618112,1.257578,2.169119,1.340758
5,Base,10997,sea,94252,233,789,0.237355,1.257578,2.185843,0.518821
6,Base,10997,ship,94252,97,600,0.510992,1.257578,2.166297,1.106961
7,Base,10997,whale,94252,17,132,2.022172,1.257578,2.020628,4.086059
8,Sin ajuste por longitud,10545,captain,170483,205,539,0.618112,1.000000,2.187197,1.351933
9,Sin ajuste por longitud,10545,sea,170483,253,789,0.237355,1.000000,2.189614,0.519716


In [20]:
tabla_posiciones = tabla_parametros.pivot(
    index="Identificador", columns="Configuración", values="Posición"
).reindex(columns=[nombre for nombre, _, _ in configuraciones])
display(tabla_posiciones)

tabla_cambios_aportes = tabla_aportes.pivot(
    index=["Identificador", "Término"], columns="Configuración", values="Aporte BM25"
).reindex(columns=[nombre for nombre, _, _ in configuraciones])
tabla_cambios_aportes["Diferencia sin ajuste"] = (
    tabla_cambios_aportes["Sin ajuste por longitud"] - tabla_cambios_aportes["Base"]
)
tabla_cambios_aportes["Diferencia k1=2"] = (
    tabla_cambios_aportes["Mayor efecto de las repeticiones"] - tabla_cambios_aportes["Base"]
)
display(tabla_cambios_aportes)

Configuración,Base,Sin ajuste por longitud,Mayor efecto de las repeticiones
Identificador,,,
10545,1,1,1
10997,3,3,3


Configuración              Base  Sin ajuste por longitud  \
Identificador Término                                      
10545         captain  1.343547                 1.351933   
              sea      0.517098                 0.519716   
              ship     0.987713                 1.053921   
              whale    4.265861                 4.358601   
10997         captain  1.340758                 1.344624   
              sea      0.518821                 0.519505   
              ship     1.106961                 1.110445   
              whale    4.086059                 4.155453   

Configuración          Mayor efecto de las repeticiones  \
Identificador Término                                     
10545         captain                          1.817585   
              sea                              0.700587   
              ship                             1.246040   
              whale                            5.661886   
10997         captain                          1.811356   
              sea                              0.704460   
              ship                             1.494232   
              whale                            5.284651   

Configuración          Diferencia sin ajuste  Diferencia k1=2  
Identificador Término                                          
10545         captain               0.008386         0.474038  
              sea                   0.002618         0.183489  
              ship                  0.066208         0.258327  
              whale                 0.092740         1.396025  
10997         captain               0.003866         0.470598  
              sea                   0.000685         0.185640  
              ship                  0.003484         0.387271  
              whale                 0.069394         1.198592

In [21]:
for doc_id in ids_documentos[list(seleccionados)]:
    datos = tabla_parametros[tabla_parametros["Identificador"] == doc_id].set_index("Configuración")
    base = datos.loc["Base"]
    sin_ajuste = datos.loc["Sin ajuste por longitud"]
    mayor_k1 = datos.loc["Mayor efecto de las repeticiones"]
    display(Markdown(
        f"**Documento {doc_id}.** En la base ocupa la posición {int(base['Posición'])} "
        f"con score {base['Score BM25']:.6f}. Con b = 0 ocupa la posición "
        f"{int(sin_ajuste['Posición'])} y obtiene {sin_ajuste['Score BM25']:.6f}. "
        f"Con k1 = 2 ocupa la posición {int(mayor_k1['Posición'])} "
        f"y obtiene {mayor_k1['Score BM25']:.6f}."
    ))

display(Markdown(
    "Con **b = 0** el ajuste de longitud es 1 para todos los documentos: desaparece la "
    "ventaja relativa de los textos cortos y la penalización relativa de los largos. "
    "Con **k1 = 2** la saturación es menos rápida y las repeticiones pueden aportar más; "
    "el cambio concreto depende de TF y del ajuste de longitud, como muestran los aportes. "
    "Las posiciones se calculan frente a toda la colección, conservando los mismos dos documentos. "
    "Una posición puede mantenerse aunque cambie el score si el documento no supera ni es "
    "superado por sus vecinos. En las tres configuraciones se mantienen fijos los tokens, "
    "DF, IDF y la longitud media; solo cambian los parámetros de BM25."
))

**Documento 10545.** En la base ocupa la posición 1 con score 7.114219. Con b = 0 ocupa la posición 1 y obtiene 7.284170. Con k1 = 2 ocupa la posición 1 y obtiene 9.426098.

**Documento 10997.** En la base ocupa la posición 3 con score 7.052598. Con b = 0 ocupa la posición 3 y obtiene 7.130027. Con k1 = 2 ocupa la posición 3 y obtiene 9.294699.

Con **b = 0** el ajuste de longitud es 1 para todos los documentos: desaparece la ventaja relativa de los textos cortos y la penalización relativa de los largos. Con **k1 = 2** la saturación es menos rápida y las repeticiones pueden aportar más; el cambio concreto depende de TF y del ajuste de longitud, como muestran los aportes. Las posiciones se calculan frente a toda la colección, conservando los mismos dos documentos. Una posición puede mantenerse aunque cambie el score si el documento no supera ni es superado por sus vecinos. En las tres configuraciones se mantienen fijos los tokens, DF, IDF y la longitud media; solo cambian los parámetros de BM25.

Los dos documentos superan la longitud media. Al quitar el ajuste por longitud, el 10545 gana 0.169952 puntos y el 10997 gana 0.077429: retirar la penalización beneficia más al documento más largo. El aporte de `whale` pasa de 4.265861 a 4.358601 en el primero y de 4.086059 a 4.155453 en el segundo.

Con k1 = 2, el aporte de `whale` aumenta a 5.661886 y 5.284651, respectivamente. Aunque `ship` favorece al 10997, esa ventaja no compensa los demás aportes del 10545. La diferencia de puntuación entre ambos es positiva en las tres configuraciones: 0.061621, 0.154143 y 0.131399. Cambian los aportes y las puntuaciones, pero se conserva su orden relativo; los rankings sobre toda la colección confirman que tampoco cruzan a sus vecinos y permanecen en las posiciones 1 y 3.